# Variant extraction inference and tokenizer-addition comparison

This notebook runs the same LLM-style variant extraction prompt on `LLM_evaluation_statistics.csv`, adds prediction columns to the file, evaluates predictions against `Human`, then repeats inference after adding domain-specific variant tokens to the tokenizer.

The final section compares tokenization for error cases before and after token addition.

**Important:** adding tokens to a causal LM tokenizer and resizing embeddings creates new, randomly initialized embeddings unless the model is fine-tuned afterward. This notebook measures the inference effect of token addition, but token addition alone is not expected to reliably improve extraction quality.

In [1]:
# If needed, uncomment and run once:
# !pip install pandas numpy matplotlib python-dotenv transformers accelerate torch scikit-learn tqdm

import os
import re
import gc
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from dotenv import load_dotenv
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import confusion_matrix, f1_score

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 120)

load_dotenv()
print("Imports loaded.")

Imports loaded.


## 1. Configuration

Set `INPUT_CSV` to the file with `PaperTitle`, `Abstract`, and `Human`. The notebook writes an augmented CSV and evaluation outputs to `OUTPUT_DIR`.

`MODEL_NAME` must be a Hugging Face causal-LM checkpoint that you can load locally. Token addition cannot be applied to a remote API model such as DeepInfra/OpenAI without retraining or hosting a modified model.

In [2]:
# -----------------------------
# Paths
# -----------------------------
INPUT_CSV = Path(os.getenv("INPUT_CSV", "LLM_evaluation_statistics.csv"))
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "variant_extraction_token_added_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Model and inference settings
# -----------------------------
# Use a local Hugging Face model. Change this to your available checkpoint.
MODEL_NAME = os.getenv("HF_MODEL_NAME", "meta-llama/Llama-3.1-8B-Instruct")
SYSTEM_MSG = "You are a helpful medical question answering assistant. Please carefully follow the exact instructions and do not provide explanations."

MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "96"))
DO_SAMPLE = False
TEMPERATURE = None

# Use SAMPLE_N_ROWS for a quick smoke test; set to None for full inference.
SAMPLE_N_ROWS = None  # e.g., 10

# Output columns to create in the augmented CSV.
BASELINE_COL = "NER_without_added_tokens"
ADDED_TOKEN_COL = "NER_with_added_tokens"

print("INPUT_CSV:", INPUT_CSV.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("MODEL_NAME:", MODEL_NAME)

INPUT_CSV: /Users/andreiblindu/Desktop/Idiap/code/Variantscape/notebooks/05_LLM_variant_extraction/LLM_evaluation_statistics.csv
OUTPUT_DIR: /Users/andreiblindu/Desktop/Idiap/code/Variantscape/notebooks/05_LLM_variant_extraction/variant_extraction_token_added_outputs
MODEL_NAME: meta-llama/Llama-3.1-8B-Instruct


## 2. Load and validate the evaluation file

In [3]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Could not find input CSV: {INPUT_CSV.resolve()}")

df = pd.read_csv(INPUT_CSV)

required_cols = ["PaperTitle", "Abstract", "Human"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if "PaperId" not in df.columns:
    df.insert(0, "PaperId", np.arange(len(df)))
    print("No PaperId column found; created a row-index PaperId.")

if SAMPLE_N_ROWS is not None:
    df = df.head(SAMPLE_N_ROWS).copy()

print(f"Loaded {len(df):,} rows")
display(df[["PaperId", "PaperTitle", "Abstract", "Human"]].head())

Loaded 797 rows


,PaperId,PaperTitle,Abstract,Human
0,4403747873,Germline DNA damage repair variants and prognosis of patients with high-risk or metastatic prostate cancer,Abstract Purpose: Deleterious germline variants in certain DNA repair genes are risk factors for developing aggressive prostate cancer. The objective was to quantify their prognostic impact after prostate cancer diag...,0
1,4404066987,The Current Status of Comprehensive Genomic Profiling in the Management of Metastatic Castration-Resistant Prostate Cancer: A Study from a Cooperative Hospital for Cancer Genomic Medicine in Japan,"Background: Several effective treatment modalities against metastatic castration-resistant prostate cancer (mCRPC) are available; however, an unmet clinical need persists for mCRPC treatment because resistance to the...",0
2,4403382046,Skin metastasis of BRCA mutated prostate cancer: A case report and a brief review of literature,"Rationale: Metastatic castration-resistant prostate cancer has a poor prognosis especially when harboring DNA damage repair gene mutations, nevertheless, in the case of pathogenic BRCA gene mutations, PARPi demonstra...",0
3,4403130090,"CDK12 loss drives prostate cancer progression, transcription-replication conflicts, and synthetic lethality with paralog CDK13","Biallelic loss of cyclin-dependent kinase 12 (CDK12) defines a metastatic castration-resistant prostate cancer (mCRPC) subtype. It remains unclear, however, whether CDK12 loss drives prostate cancer (PCa) development...",0
4,4402986712,TMPRSS2:ERGGene Fusion Might Predict Resistance to PARP Inhibitors in Metastatic Castration-resistant Prostate Cancer,The emergence of novel DNA damage repair (DDR) pathways in molecular-target therapy drugs (MTTD) has shown promising outcomes in treating patients with metastatic castration-resistant prostate cancer (mCRPC). About 2...,0


## 3. Prompt used for extraction

This is the compact prompt from the LLM-based extraction notebook.

In [4]:
def build_variant_prompt(title, abstract):
    return (
        "Extract only specific genetic variants from the text. Return strictly:\n"
        "- **HGVS Notation** (c., p., g.) e.g., c.2138C>G, p.Arg713Trp\n"
        "- **Protein changes** (e.g., V600E, Arg713Trp)\n"
        "- **rsIDs** (e.g., rs121913529)\n"
        "- Ignore vague terms (e.g., 'mutation found').\n\n"
        "### Format:\n"
        "- Variant: 'Variant: <mutation>, Gene: <gene>' per line\n"
        "- If none, return: 'No variant'\n"
        "- No extra text, no explanations.\n\n"
        f"Title: {title}\nAbstract: {abstract}"
    )

print(build_variant_prompt("Example title", "Example abstract mentioning BRAF V600E.")[:600])

Extract only specific genetic variants from the text. Return strictly:
- **HGVS Notation** (c., p., g.) e.g., c.2138C>G, p.Arg713Trp
- **Protein changes** (e.g., V600E, Arg713Trp)
- **rsIDs** (e.g., rs121913529)
- Ignore vague terms (e.g., 'mutation found').

### Format:
- Variant: 'Variant: <mutation>, Gene: <gene>' per line
- If none, return: 'No variant'
- No extra text, no explanations.

Title: Example title
Abstract: Example abstract mentioning BRAF V600E.


## 4. Model loading and inference helpers

In [5]:
def load_causal_lm(model_name, add_tokens=False, new_tokens=None):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True,
        trust_remote_code=True,
        token=os.getenv("HF_TOKEN")
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        token=os.getenv("HF_TOKEN")
    )

    n_added = 0
    if add_tokens:
        n_added = tokenizer.add_tokens(new_tokens or [])
        if n_added > 0:
            model.resize_token_embeddings(len(tokenizer))
            model.config.vocab_size = len(tokenizer)

    model.eval()
    return tokenizer, model, n_added


def format_chat_prompt(prompt, tokenizer):
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"System: {SYSTEM_MSG}\nUser: {prompt}\nAssistant:"


@torch.inference_mode()
def generate_from_prompt(prompt, tokenizer, model):
    model_input = format_chat_prompt(prompt, tokenizer)
    inputs = tokenizer(model_input, return_tensors="pt", truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    generate_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if TEMPERATURE is not None:
        generate_kwargs["temperature"] = TEMPERATURE

    output_ids = model.generate(**inputs, **generate_kwargs)
    new_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def run_inference(dataframe, tokenizer, model, output_col):
    predictions = []
    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc=output_col):
        prompt = build_variant_prompt(row["PaperTitle"], row["Abstract"])
        try:
            predictions.append(generate_from_prompt(prompt, tokenizer, model))
        except Exception as exc:
            predictions.append(f"ERROR: {type(exc).__name__}: {exc}")
    dataframe[output_col] = predictions
    return dataframe

## 5. Baseline inference without added tokens

In [6]:
tokenizer, model, n_added = load_causal_lm(MODEL_NAME, add_tokens=False)
print("Loaded baseline model. Added tokens:", n_added)

df = run_inference(df, tokenizer, model, BASELINE_COL)

display(df[["PaperId", "Human", BASELINE_COL]].head())

# Free memory before loading the token-extended model.
del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded baseline model. Added tokens: 0


NER_without_added_tokens:   0%|          | 0/797 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

## 6. Inference after adding variant-specific tokens

In [ ]:
new_tokens = [
    "c.", "p.", "g.", "m.", "n.", "r.",
    "del", "dup", "ins", "delins", "fs", "Ter",
    "rs",
    "Ala", "Arg", "Asn", "Asp", "Cys", "Gln", "Glu", "Gly",
    "His", "Ile", "Leu", "Lys", "Met", "Phe", "Pro", "Ser",
    "Thr", "Trp", "Tyr", "Val",
    ">", "_", ":",
    "BRCA1", "BRAF", "KRAS"
]

tokenizer, model, n_added = load_causal_lm(MODEL_NAME, add_tokens=True, new_tokens=new_tokens)
print(f"Requested {len(new_tokens)} tokens; actually added {n_added} new tokens.")

df = run_inference(df, tokenizer, model, ADDED_TOKEN_COL)

display(df[["PaperId", "Human", BASELINE_COL, ADDED_TOKEN_COL]].head())

# Keep this tokenizer for immediate inspection if desired, then free the model.
del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. Evaluation against the human ground truth

For compatibility with the original evaluation notebook, `Human == "0"` is treated as negative and anything else is treated as positive. Model outputs are negative only when they are empty or explicitly say no variant.

In [ ]:
def human_binary_label(x):
    if pd.isna(x):
        return 0
    return 0 if str(x).strip() == "0" else 1


def prediction_binary_label(x):
    if pd.isna(x):
        return 0
    text = str(x).strip().lower()
    if text == "" or text.startswith("error:"):
        return 0
    no_variant_patterns = [
        "0", "none", "nan", "no variant", "no variants",
        "no genetic variant", "no genetic variants",
        "no genetic variant detected",
        "no genetic variant detected in this publication",
    ]
    return 0 if any(text == p or text.startswith(p + ".") for p in no_variant_patterns) else 1


def compute_binary_metrics(dataframe, pred_col, gold_col="Human"):
    y_true = dataframe[gold_col].map(human_binary_label)
    y_pred = dataframe[pred_col].map(prediction_binary_label)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    accuracy = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) else np.nan
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return {
        "model": pred_col,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "accuracy": accuracy,
        "f1": f1,
    }

comparison_cols = [BASELINE_COL, ADDED_TOKEN_COL]
metrics_df = pd.DataFrame([compute_binary_metrics(df, c) for c in comparison_cols])

display(metrics_df)

# Bar chart: compare main metrics.
plot_df = metrics_df.set_index("model")[["precision", "recall", "f1", "accuracy"]]
ax = plot_df.plot(kind="bar", figsize=(9, 4), rot=20)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("NER extraction performance: without vs with added tokens")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 8. Error tables

In [ ]:
def add_error_type(dataframe, pred_col, gold_col="Human"):
    out = dataframe.copy()
    out["y_true"] = out[gold_col].map(human_binary_label)
    out["y_pred"] = out[pred_col].map(prediction_binary_label)
    conditions = [
        (out["y_true"].eq(1) & out["y_pred"].eq(1)),
        (out["y_true"].eq(0) & out["y_pred"].eq(1)),
        (out["y_true"].eq(1) & out["y_pred"].eq(0)),
        (out["y_true"].eq(0) & out["y_pred"].eq(0)),
    ]
    choices = ["TP", "FP", "FN", "TN"]
    out["error_type"] = np.select(conditions, choices, default="UNKNOWN")
    out["prediction_column"] = pred_col
    return out

baseline_error_df = add_error_type(df, BASELINE_COL)
added_error_df = add_error_type(df, ADDED_TOKEN_COL)
all_error_df = pd.concat([baseline_error_df, added_error_df], ignore_index=True)

display(all_error_df.groupby(["prediction_column", "error_type"]).size().rename("count").to_frame())

display(
    all_error_df[all_error_df["error_type"].isin(["FP", "FN"])]
    [["prediction_column", "error_type", "PaperId", "PaperTitle", "Human", BASELINE_COL, ADDED_TOKEN_COL]]
    .head(50)
)

## 9. Tokenization analysis for error cases

This follows the second notebook: extract variant-like candidate strings from titles and abstracts in error rows, then compare tokenization before and after adding the new tokens.

In [ ]:
VARIANT_REGEX = re.compile(
    r"""
    (?:
        rs\d+
        |
        [cpgnmr]\.\d+(?:_\d+)?(?:[ACGT]>[ACGT]|del[A-Za-z0-9]*|dup[A-Za-z0-9]*|ins[A-Za-z0-9]*|delins[A-Za-z0-9]*)
        |
        p\.[A-Z][a-z]{2}\d+[A-Z][a-z]{2}
        |
        p\.[A-Z]\d+[A-Z]
        |
        N[MR]_\d+(?:\.\d+)?:[cpgnmr]\.[A-Za-z0-9_>.+\-]+
        |
        [A-Z]{1,8}\s+[A-Z]\d+[A-Z]
        |
        [A-Z]\d+[A-Z]
        |
        \d+(?:del|ins|dup)[A-Za-z]*
        |
        IVS\d+[+\-]\d+[ACGT]>[ACGT]
    )
    """,
    flags=re.VERBOSE | re.IGNORECASE,
)


def extract_candidate_variants(text):
    if pd.isna(text):
        return []
    text = str(text)
    candidates = [m.group(0).strip() for m in VARIANT_REGEX.finditer(text)]
    return sorted({c for c in candidates if len(c) >= 3})


def tokenization_features(variant, tokenizer):
    variant = str(variant).strip()
    result = {
        "variant": variant,
        "n_chars": len(variant),
        "has_punctuation": bool(re.search(r"[.\->_/+:]", variant)),
        "has_digit": bool(re.search(r"\d", variant)),
        "has_mixed_case": bool(re.search(r"[a-z]", variant) and re.search(r"[A-Z]", variant)),
        "has_hgvs_like_prefix": bool(re.search(r"[cpgnmr]\.", variant, flags=re.IGNORECASE)),
        "has_reference_sequence": bool(re.search(r"N[MR]_\d+(?:\.\d+)?", variant)),
        "has_rs_id": bool(re.search(r"rs\d+", variant, flags=re.IGNORECASE)),
    }

    enc = tokenizer(variant, add_special_tokens=False, return_offsets_mapping=True)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    pieces = [variant[s:e] for s, e in enc["offset_mapping"]]

    result["n_tokens_raw"] = len(tokens)
    result["chars_per_token_raw"] = len(variant) / len(tokens) if tokens else np.nan
    result["fragmentation_ratio"] = len(tokens) / max(len(variant), 1)
    result["tokens_raw"] = tokens
    result["pieces_raw"] = pieces
    return result


def inspect_tokenization(text, tokenizer):
    text = str(text)
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    rows = []
    for i, (tok, tok_id, offset) in enumerate(zip(tokens, enc["input_ids"], enc["offset_mapping"])):
        start, end = offset
        rows.append({
            "token_index": i,
            "token": tok,
            "token_id": tok_id,
            "char_start": start,
            "char_end": end,
            "text_piece": text[start:end],
        })
    return pd.DataFrame(rows)


def load_analysis_tokenizers():
    tok_base = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True, token=os.getenv("HF_TOKEN"))
    tok_added = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True, token=os.getenv("HF_TOKEN"))
    n_added_for_analysis = tok_added.add_tokens(new_tokens)
    print(f"Analysis tokenizer added tokens: {n_added_for_analysis}")
    return {"without_added_tokens": tok_base, "with_added_tokens": tok_added}

analysis_tokenizers = load_analysis_tokenizers()

In [ ]:
error_rows = all_error_df[all_error_df["error_type"].isin(["FP", "FN"])].copy()

candidate_rows = []
for _, row in error_rows.iterrows():
    text = f"{row.get('PaperTitle', '')} {row.get('Abstract', '')}"
    candidates = extract_candidate_variants(text)
    for cand in candidates:
        for tokenizer_label, tok in analysis_tokenizers.items():
            feats = tokenization_features(cand, tok)
            feats.update({
                "prediction_column": row["prediction_column"],
                "error_type": row["error_type"],
                "PaperId": row["PaperId"],
                "candidate_variant": cand,
                "tokenizer_version": tokenizer_label,
                "PaperTitle": row.get("PaperTitle", None),
            })
            candidate_rows.append(feats)

candidate_tok_df = pd.DataFrame(candidate_rows)
print(f"Candidate variant/tokenizer rows: {len(candidate_tok_df):,}")

display(candidate_tok_df.head())

if len(candidate_tok_df) > 0:
    token_summary = (
        candidate_tok_df
        .groupby(["prediction_column", "error_type", "tokenizer_version"])
        .agg(
            n_candidate_mentions=("candidate_variant", "count"),
            n_articles=("PaperId", "nunique"),
            mean_tokens=("n_tokens_raw", "mean"),
            median_tokens=("n_tokens_raw", "median"),
            mean_fragmentation=("fragmentation_ratio", "mean"),
            pct_hgvs_like=("has_hgvs_like_prefix", "mean"),
            pct_reference_sequence=("has_reference_sequence", "mean"),
            pct_rs_id=("has_rs_id", "mean"),
        )
        .reset_index()
        .sort_values(["prediction_column", "error_type", "tokenizer_version"])
    )
    display(token_summary)
else:
    print("No candidate variants found in FP/FN rows by the regex.")

## 10. Inspect tokenization for the most fragmented error candidates

In [ ]:
def show_most_fragmented_errors(prediction_column=BASELINE_COL, error_type="FN", top_n=20):
    if len(candidate_tok_df) == 0:
        print("No candidate tokenization data available.")
        return pd.DataFrame()

    subset = candidate_tok_df[
        candidate_tok_df["prediction_column"].eq(prediction_column)
        & candidate_tok_df["error_type"].eq(error_type)
    ].copy()

    if subset.empty:
        print(f"No rows for prediction_column={prediction_column}, error_type={error_type}")
        return subset

    wide = (
        subset
        .pivot_table(
            index=["PaperId", "candidate_variant", "PaperTitle"],
            columns="tokenizer_version",
            values=["n_tokens_raw", "fragmentation_ratio"],
            aggfunc="first"
        )
    )
    wide.columns = ["_".join(col).strip() for col in wide.columns.to_flat_index()]
    wide = wide.reset_index()

    sort_col = "n_tokens_raw_without_added_tokens"
    if sort_col not in wide.columns:
        sort_col = [c for c in wide.columns if c.startswith("n_tokens_raw")][0]

    display(wide.sort_values(sort_col, ascending=False).head(top_n))
    return wide

fragmented_fn = show_most_fragmented_errors(BASELINE_COL, "FN", top_n=25)

In [ ]:
# Choose a variant from the table above or set one manually.
VARIANT_TO_INSPECT = None

if VARIANT_TO_INSPECT is None and len(candidate_tok_df) > 0:
    fn_candidates = candidate_tok_df[
        candidate_tok_df["error_type"].eq("FN")
        & candidate_tok_df["tokenizer_version"].eq("without_added_tokens")
    ]
    if len(fn_candidates) > 0:
        VARIANT_TO_INSPECT = (
            fn_candidates
            .sort_values(["n_tokens_raw", "fragmentation_ratio"], ascending=False)
            .iloc[0]["candidate_variant"]
        )

if VARIANT_TO_INSPECT is not None:
    print("Inspecting:", VARIANT_TO_INSPECT)
    for label, tok in analysis_tokenizers.items():
        print("\nTokenizer:", label)
        display(inspect_tokenization(VARIANT_TO_INSPECT, tok))
else:
    print("No variant available to inspect.")

## 11. Export augmented file and analysis outputs

In [ ]:
augmented_out = OUTPUT_DIR / "LLM_evaluation_statistics_with_NER_token_added_predictions.csv"
metrics_out = OUTPUT_DIR / "NER_without_vs_with_added_tokens_metrics.csv"
errors_out = OUTPUT_DIR / "NER_without_vs_with_added_tokens_error_rows.csv"
tokenization_out = OUTPUT_DIR / "NER_error_candidate_tokenization_without_vs_with_added_tokens.csv"

# Add binary helper columns for easier downstream checks.
df[f"{BASELINE_COL}_binary"] = df[BASELINE_COL].map(prediction_binary_label)
df[f"{ADDED_TOKEN_COL}_binary"] = df[ADDED_TOKEN_COL].map(prediction_binary_label)
df["Human_binary"] = df["Human"].map(human_binary_label)

df.to_csv(augmented_out, index=False)
metrics_df.to_csv(metrics_out, index=False)
all_error_df.to_csv(errors_out, index=False)
if "candidate_tok_df" in globals() and len(candidate_tok_df) > 0:
    candidate_tok_df.to_csv(tokenization_out, index=False)

print("Saved:")
print("-", augmented_out.resolve())
print("-", metrics_out.resolve())
print("-", errors_out.resolve())
if "candidate_tok_df" in globals() and len(candidate_tok_df) > 0:
    print("-", tokenization_out.resolve())